In [ ]:
!pip install sentence-transformers torch torch-geometric torch-scatter torch-sparse torch-cluster torch-spline-conv scikit-learn networkx matplotlib pandas numpy

Install PyTorch Geometric using pre-built wheels (Fastest)

In [ ]:
# First install PyTorch (if not already installed)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118  # For CUDA 11.8
# OR for CPU only:
# !pip install torch torchvision torchaudio

# Install PyTorch Geometric using pre-built wheels (much faster)
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-2.0.0+cu118.html
!pip install torch-geometric

# Install other dependencies
!pip install sentence-transformers scikit-learn networkx matplotlib pandas numpy

original packages that takes much time to get installed

In [ ]:
!pip install sentence-transformers torch torch-geometric torch-scatter torch-sparse torch-cluster torch-spline-conv scikit-learn networkx matplotlib pandas numpy


mount drive

In [ ]:
# prompt: code for mount drive

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import torch
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors
from torch_geometric.data import Data
from sklearn.preprocessing import LabelEncoder
import pickle
from torch_geometric.nn import SAGEConv
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from torch_geometric.utils import negative_sampling
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load and preprocess data for sentiment analysis
def load_sentiment_data(filepath):
    df = pd.read_csv(filepath)

    # Check for required columns
    required_cols = ["Sentences", "Label"]
    if not all(col in df.columns for col in required_cols):
        raise ValueError(f"Dataset must contain {required_cols} columns.")

    # Clean data
    df = df.dropna(subset=["Sentences"])

    # Check class distribution before encoding
    print("\nOriginal Class Distribution:")
    print(df["Label"].value_counts())

    # Filter out classes with very few samples (optional)
    class_counts = df["Label"].value_counts()
    min_samples_per_class = 5  # Minimum samples required per class

    # Remove classes with less than min_samples_per_class
    classes_to_keep = class_counts[class_counts >= min_samples_per_class].index.tolist()
    if len(classes_to_keep) < len(class_counts):
        print(f"\nWarning: Removing classes with less than {min_samples_per_class} samples:")
        removed_classes = [c for c in class_counts.index if c not in classes_to_keep]
        print(f"Removed: {removed_classes}")
        df = df[df["Label"].isin(classes_to_keep)]
        print(f"Remaining samples: {len(df)}")

    # Encode sentiment labels
    label_encoder = LabelEncoder()
    df["Label_encoded"] = label_encoder.fit_transform(df["Label"])

    print("\nClass Distribution after filtering:")
    print(df["Label"].value_counts())
    print(f"\nLabel mapping: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")

    return df, label_encoder

# Graph construction for sentiment data
def build_sentiment_graph(texts, labels, k=15, threshold=0.6):
    # Generate embeddings with progress bar
    model = SentenceTransformer("all-mpnet-base-v2", device=device)
    print("\nGenerating sentence embeddings...")
    embeddings = model.encode(texts,
                            batch_size=128,
                            show_progress_bar=True,
                            convert_to_tensor=True,
                            device=device)
    embeddings = embeddings.cpu().numpy()

    # Build KNN graph
    print("\nBuilding KNN graph...")
    nbrs = NearestNeighbors(n_neighbors=min(k, len(texts)-1), metric="cosine").fit(embeddings)
    distances, indices = nbrs.kneighbors(embeddings)

    # Create graph with node attributes
    G = nx.Graph()
    for i, (text, label) in enumerate(zip(texts, labels)):
        G.add_node(i, text=text, label=label)

    # Add edges with similarity filtering
    edge_count = 0
    for i in range(len(texts)):
        for j, sim_score in zip(indices[i], distances[i]):
            if i != j:  # Avoid self-loops
                similarity = 1 - sim_score
                if similarity > threshold:
                    G.add_edge(i, j, weight=similarity)
                    edge_count += 1

    print(f"\nGraph statistics:")
    print(f"- Nodes: {G.number_of_nodes()}")
    print(f"- Edges: {G.number_of_edges()}")
    print(f"- Average degree: {2 * G.number_of_edges() / G.number_of_nodes():.2f}")

    return G, embeddings

# Enhanced GraphSAGE model for multi-class sentiment classification
class SentimentGraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels=256, num_classes=3):
        super().__init__()
        # First SAGE layer
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.bn1 = torch.nn.BatchNorm1d(hidden_channels)

        # Second SAGE layer
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.bn2 = torch.nn.BatchNorm1d(hidden_channels)

        # Third SAGE layer
        self.conv3 = SAGEConv(hidden_channels, hidden_channels//2)
        self.bn3 = torch.nn.BatchNorm1d(hidden_channels//2)

        # Final classification layer for multi-class
        self.fc = torch.nn.Linear(hidden_channels//2, num_classes)

        # Regularization
        self.dropout = torch.nn.Dropout(0.3)

    def forward(self, x, edge_index):
        # First block
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout(x)

        # Second block
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.dropout(x)

        # Third block
        x = self.conv3(x, edge_index)
        x = self.bn3(x)
        x = F.relu(x)

        # Final classification
        x = self.fc(x)
        return x

# Function to create stratified splits with handling for small classes
def create_stratified_splits(labels, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, random_state=42):
    """
    Create stratified splits even with small class sizes
    """
    labels_np = np.array(labels)
    n_samples = len(labels_np)

    # Check class distribution
    class_counts = Counter(labels_np)
    print("\nClass distribution for splitting:")
    for class_label, count in class_counts.items():
        print(f"  Class {class_label}: {count} samples")

    # Check if any class has very few samples
    min_class_size = min(class_counts.values())

    if min_class_size < 2:
        print(f"\nWarning: Class with only {min_class_size} sample(s) detected!")
        print("Using non-stratified splitting to avoid error...")

        # Use simple random split without stratification
        indices = np.arange(n_samples)

        # First split: train and temp
        train_idx, temp_idx = train_test_split(
            indices,
            test_size=(val_ratio + test_ratio),
            random_state=random_state
        )

        # Second split: val and test from temp
        val_ratio_adjusted = val_ratio / (val_ratio + test_ratio)
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1-val_ratio_adjusted,
            random_state=random_state
        )

    elif min_class_size < 3:
        print(f"\nWarning: Class with only {min_class_size} samples!")
        print("Using StratifiedShuffleSplit with multiple splits...")

        # Use StratifiedShuffleSplit for better handling
        indices = np.arange(n_samples)

        # First, create train/test split
        sss1 = StratifiedShuffleSplit(n_splits=1, test_size=(val_ratio + test_ratio), random_state=random_state)
        train_idx, temp_idx = next(sss1.split(indices, labels_np))

        # Then split temp into val and test
        temp_labels = labels_np[temp_idx]
        val_ratio_adjusted = val_ratio / (val_ratio + test_ratio)

        # If the class is too small in temp, use simple split
        if min(Counter(temp_labels).values()) < 2:
            val_idx, test_idx = train_test_split(
                temp_idx,
                test_size=0.5,
                random_state=random_state
            )
        else:
            sss2 = StratifiedShuffleSplit(n_splits=1, test_size=1-val_ratio_adjusted, random_state=random_state)
            val_idx, test_idx = next(sss2.split(temp_idx, temp_labels))
            val_idx = temp_idx[val_idx]
            test_idx = temp_idx[test_idx]

    else:
        # Normal stratified splitting
        indices = np.arange(n_samples)

        # First split: train and temp
        train_idx, temp_idx = train_test_split(
            indices,
            test_size=(val_ratio + test_ratio),
            stratify=labels_np,
            random_state=random_state
        )

        # Second split: val and test from temp
        temp_labels = labels_np[temp_idx]
        val_ratio_adjusted = val_ratio / (val_ratio + test_ratio)

        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1-val_ratio_adjusted,
            stratify=temp_labels,
            random_state=random_state
        )

    return train_idx, val_idx, test_idx

# Training and evaluation for sentiment analysis
def train_sentiment_model(graph_data, train_idx, val_idx, test_idx, num_classes, epochs=100):
    # Initialize model
    model = SentimentGraphSAGE(
        in_channels=graph_data.num_features,
        hidden_channels=256,
        num_classes=num_classes
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=10, factor=0.5)
    criterion = torch.nn.CrossEntropyLoss()

    best_val_acc = 0
    best_model_state = None

    # Convert indices to masks
    train_mask = torch.zeros(graph_data.num_nodes, dtype=torch.bool)
    val_mask = torch.zeros(graph_data.num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(graph_data.num_nodes, dtype=torch.bool)
    train_mask[train_idx] = True
    val_mask[val_idx] = True
    test_mask[test_idx] = True

    print(f"\nTraining samples per class:")
    for i in range(num_classes):
        train_count = (graph_data.y[train_mask] == i).sum().item()
        print(f"  Class {i}: {train_count} samples")

    # Training loop
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        # Forward pass
        out = model(graph_data.x.to(device), graph_data.edge_index.to(device))

        # Classification loss
        cls_loss = criterion(out[train_mask], graph_data.y.to(device)[train_mask])

        # Optional: Add class weights for imbalanced data
        total_loss = cls_loss

        total_loss.backward()
        optimizer.step()

        # Validation
        model.eval()
        with torch.no_grad():
            out = model(graph_data.x.to(device), graph_data.edge_index.to(device))

            # Calculate validation metrics
            val_pred = out[val_mask].argmax(dim=1)
            val_acc = (val_pred == graph_data.y.to(device)[val_mask]).float().mean()

            # Save best model based on validation accuracy
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_model_state = model.state_dict().copy()

        # Print progress
        if (epoch + 1) % 20 == 0:
            train_pred = out[train_mask].argmax(dim=1)
            train_acc = (train_pred == graph_data.y.to(device)[train_mask]).float().mean()

            print(f"Epoch {epoch+1:03d}:")
            print(f"- Loss: {total_loss.item():.4f}")
            print(f"- Train Acc: {train_acc:.4f}")
            print(f"- Val Acc: {val_acc:.4f}")

    # Final evaluation on test set
    model.load_state_dict(best_model_state)
    model.eval()
    with torch.no_grad():
        out = model(graph_data.x.to(device), graph_data.edge_index.to(device))

        # Get predictions and probabilities
        test_logits = out[test_mask]
        test_pred = test_logits.argmax(dim=1).cpu().numpy()
        test_probs = F.softmax(test_logits, dim=1).cpu().numpy()
        test_true = graph_data.y[test_mask].cpu().numpy()

        # Calculate metrics
        test_acc = accuracy_score(test_true, test_pred)

        print("\n" + "="*50)
        print("Final Test Results:")
        print(f"- Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
        print("\nClassification Report:")

        # Get class names from label encoder
        class_names = label_encoder.classes_
        print(classification_report(test_true, test_pred, target_names=class_names, zero_division=0))

        print("\nConfusion Matrix:")
        print(confusion_matrix(test_true, test_pred))

        # Per-class accuracy
        print("\nPer-class Accuracy:")
        for i, class_name in enumerate(class_names):
            class_mask = test_true == i
            if class_mask.sum() > 0:
                class_acc = (test_pred[class_mask] == i).mean()
                print(f"  {class_name}: {class_acc:.4f} ({class_acc*100:.2f}%)")
            else:
                print(f"  {class_name}: No samples in test set")

        print("="*50)

    return model, best_val_acc, test_acc

# Main execution
if __name__ == "__main__":
    # Load and prepare sentiment data
    print("Loading sentiment data...")
    df, label_encoder = load_sentiment_data("/content/dataset/final_dataset_sentiment_analysis.csv")

    # Get number of classes
    num_classes = len(label_encoder.classes_)
    print(f"\nNumber of sentiment classes: {num_classes}")

    # Build graph
    print("\nBuilding graph structure...")
    G, embeddings = build_sentiment_graph(
        texts=df["Sentences"].tolist(),
        labels=df["Label_encoded"].tolist(),
        k=min(15, len(df)-1),  # Number of neighbors
        threshold=0.6  # Similarity threshold
    )

    # Convert to PyTorch Geometric format
    edge_index = torch.tensor(list(G.edges), dtype=torch.long).t().contiguous()
    node_features = torch.tensor(embeddings, dtype=torch.float32)
    node_labels = torch.tensor([G.nodes[n]["label"] for n in G.nodes], dtype=torch.long)

    graph_data = Data(x=node_features, edge_index=edge_index, y=node_labels)

    # Create stratified splits with handling for small classes
    print("\nCreating train/val/test splits...")
    train_idx, val_idx, test_idx = create_stratified_splits(
        graph_data.y.numpy(),
        train_ratio=0.7,
        val_ratio=0.15,
        test_ratio=0.15,
        random_state=42
    )

    print(f"\nSplit sizes:")
    print(f"Train set size: {len(train_idx)}")
    print(f"Validation set size: {len(val_idx)}")
    print(f"Test set size: {len(test_idx)}")

    # Verify split distribution
    print(f"\nTrain set class distribution:")
    train_labels = graph_data.y[train_idx].numpy()
    for i in range(num_classes):
        count = (train_labels == i).sum()
        print(f"  Class {i}: {count} samples")

    # Train and evaluate
    print("\nStarting training...")
    model, best_val_acc, test_acc = train_sentiment_model(
        graph_data=graph_data,
        train_idx=train_idx,
        val_idx=val_idx,
        test_idx=test_idx,
        num_classes=num_classes,
        epochs=100  # Reduced epochs for faster training
    )

    print(f"\nTraining complete!")
    print(f"Best validation accuracy: {best_val_acc:.4f}")
    print(f"Test accuracy: {test_acc:.4f}")

    # Save the model
    torch.save(model.state_dict(), "/content/drive/MyDrive/sentiment_gnn_model.pth")
    print("\nModel saved to: /content/drive/MyDrive/sentiment_gnn_model.pth")

code for saving the structured graph for your sentiment analysis dataset in .pkl format:

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import torch
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors
from torch_geometric.data import Data
import pickle
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load and preprocess sentiment data (UPDATED with filtering logic)
def load_sentiment_data(filepath):
    df = pd.read_csv(filepath)
    required_cols = ["Sentences", "Label"]
    if not all(col in df.columns for col in required_cols):
        raise ValueError(f"Dataset must contain {required_cols} columns.")

    # Clean data
    df = df.dropna(subset=["Sentences"])

    # Check class distribution before encoding
    print("\nOriginal Class Distribution:")
    print(df["Label"].value_counts())

    # Filter out classes with very few samples (to match training code)
    class_counts = df["Label"].value_counts()
    min_samples_per_class = 5  # Minimum samples required per class

    # Remove classes with less than min_samples_per_class
    classes_to_keep = class_counts[class_counts >= min_samples_per_class].index.tolist()
    if len(classes_to_keep) < len(class_counts):
        print(f"\nWarning: Removing classes with less than {min_samples_per_class} samples:")
        removed_classes = [c for c in class_counts.index if c not in classes_to_keep]
        print(f"Removed: {removed_classes}")
        df = df[df["Label"].isin(classes_to_keep)]
        print(f"Remaining samples: {len(df)}")

    # Encode sentiment labels (Positive, Negative, Neutral)
    label_encoder = LabelEncoder()
    df["Label_encoded"] = label_encoder.fit_transform(df["Label"])

    print("\nClass Distribution after filtering:")
    print(df["Label"].value_counts())
    print(f"\nLabel mapping: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")

    return df, label_encoder

# Graph construction for sentiment data (UPDATED with better error handling)
def build_sentiment_graph(texts, labels, k=15, threshold=0.6):
    model = SentenceTransformer("all-mpnet-base-v2", device=device)
    print("\nGenerating sentence embeddings...")
    embeddings = model.encode(texts,
                            batch_size=128,
                            show_progress_bar=True,
                            convert_to_tensor=True,
                            device=device)
    embeddings = embeddings.cpu().numpy()

    # Adjust k if it's larger than number of samples
    k = min(k, len(texts) - 1)

    print("\nBuilding KNN graph...")
    nbrs = NearestNeighbors(n_neighbors=k, metric="cosine").fit(embeddings)
    distances, indices = nbrs.kneighbors(embeddings)

    G = nx.Graph()
    for i, (text, label) in enumerate(zip(texts, labels)):
        G.add_node(i, text=text, label=label)

    # Add edges with similarity filtering
    edge_count = 0
    for i in range(len(texts)):
        for j, sim_score in zip(indices[i], distances[i]):
            if i != j:  # Avoid self-loops
                similarity = 1 - sim_score
                if similarity > threshold:
                    G.add_edge(i, j, weight=similarity)
                    edge_count += 1

    print(f"\nGraph statistics:")
    print(f"- Nodes: {G.number_of_nodes()}")
    print(f"- Edges: {G.number_of_edges()}")
    print(f"- Average degree: {2 * G.number_of_edges() / G.number_of_nodes():.2f}")

    return G, embeddings

# Main execution for graph construction and saving
if __name__ == "__main__":
    # Load sentiment dataset
    print("Loading sentiment data...")
    filepath = "/content/dataset/final_dataset_sentiment_analysis.csv"  # Update this path
    df, label_encoder = load_sentiment_data(filepath)

    # Get number of classes
    num_classes = len(label_encoder.classes_)
    print(f"\nNumber of sentiment classes: {num_classes}")

    # Build graph
    print("\nBuilding graph structure...")
    G, embeddings = build_sentiment_graph(
        texts=df["Sentences"].tolist(),
        labels=df["Label_encoded"].tolist(),
        k=min(15, len(df)-1),  # Number of neighbors (adjust as needed)
        threshold=0.6  # Similarity threshold (match training code)
    )

    # Convert to PyTorch Geometric format
    edge_index = torch.tensor(list(G.edges), dtype=torch.long).t().contiguous()
    node_features = torch.tensor(embeddings, dtype=torch.float32)
    node_labels = torch.tensor([G.nodes[n]["label"] for n in G.nodes], dtype=torch.long)

    # Create PyG Data object
    graph_data = Data(x=node_features, edge_index=edge_index, y=node_labels)

    # Add additional metadata
    graph_data.num_classes = num_classes
    graph_data.class_names = label_encoder.classes_.tolist()
    graph_data.texts = df["Sentences"].tolist()

    # Add label distribution for reference
    graph_data.label_distribution = df["Label"].value_counts().to_dict()

    # Save processed data
    # Option 1: Save to Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    save_path = "/content/drive/MyDrive/sentiment_graph_data.pkl"

    # Option 2: Save locally (comment out if using Drive)
    # save_path = "/content/sentiment_graph_data.pkl"

    with open(save_path, "wb") as f:
        pickle.dump(graph_data, f)

    print(f"\nGraph data saved to: {save_path}")

    # Display saved data information
    print("\n" + "="*60)
    print("SAVED GRAPH DATA INFORMATION")
    print("="*60)
    print(f"- Number of nodes: {graph_data.num_nodes}")
    print(f"- Number of edges: {graph_data.num_edges}")
    print(f"- Node features shape: {graph_data.x.shape}")
    print(f"- Edge index shape: {graph_data.edge_index.shape}")
    print(f"- Number of classes: {graph_data.num_classes}")
    print(f"- Class labels: {graph_data.class_names}")
    print(f"- Label distribution:")
    for i, class_name in enumerate(graph_data.class_names):
        count = (graph_data.y == i).sum().item()
        percentage = (count / graph_data.num_nodes) * 100
        print(f"  {class_name}: {count} nodes ({percentage:.2f}%)")

    # Show sample of texts (first 5)
    print(f"\n- Sample texts (first 5):")
    for i in range(min(5, len(graph_data.texts))):
        text_preview = graph_data.texts[i][:100] + "..." if len(graph_data.texts[i]) > 100 else graph_data.texts[i]
        label = graph_data.class_names[graph_data.y[i].item()]
        print(f"  {i+1}. [{label}] {text_preview}")

    print("="*60)

# Optional: Code to load and verify the saved graph
def load_saved_graph(filepath):
    """Function to load and verify the saved graph data"""
    with open(filepath, "rb") as f:
        loaded_data = pickle.load(f)

    print("\n" + "="*50)
    print("LOADED GRAPH DATA VERIFICATION")
    print("="*50)
    print(f"- Number of nodes: {loaded_data.num_nodes}")
    print(f"- Number of edges: {loaded_data.num_edges}")
    print(f"- Node features shape: {loaded_data.x.shape}")
    print(f"- Labels shape: {loaded_data.y.shape}")
    print(f"- Number of classes: {loaded_data.num_classes}")
    print(f"- Class names: {loaded_data.class_names}")
    print(f"- Text samples available: {len(loaded_data.texts) if hasattr(loaded_data, 'texts') else 0}")
    print("="*50)

    return loaded_data

# Example of loading the saved graph (uncomment to test)
# loaded_graph = load_saved_graph("/content/drive/MyDrive/sentiment_graph_data.pkl")

# Additional function to get graph statistics summary
def get_graph_statistics(graph_data):
    """Get detailed statistics about the graph"""
    print("\n" + "="*50)
    print("GRAPH STATISTICS SUMMARY")
    print("="*50)
    print(f"Nodes: {graph_data.num_nodes}")
    print(f"Edges: {graph_data.num_edges}")
    print(f"Average degree: {2 * graph_data.num_edges / graph_data.num_nodes:.2f}")
    print(f"Feature dimension: {graph_data.x.shape[1]}")
    print(f"Classes: {graph_data.num_classes}")
    print(f"Class distribution:")
    for i, class_name in enumerate(graph_data.class_names):
        count = (graph_data.y == i).sum().item()
        print(f"  {class_name}: {count} ({count/graph_data.num_nodes*100:.1f}%)")

    # Calculate graph density
    n = graph_data.num_nodes
    max_possible_edges = n * (n - 1) / 2
    density = graph_data.num_edges / max_possible_edges
    print(f"Graph density: {density:.6f} ({density*100:.4f}%)")
    print("="*50)

# Uncomment to see statistics
# get_graph_statistics(graph_data)

graph


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from torch_geometric.utils import to_networkx
import numpy as np
import torch

def visualize_sentiment_graph(graph_data, model, sample_size=1000):
    """
    Visualize the graph with sentiment nodes (3 classes: Positive, Negative, Neutral)

    Args:
        graph_data: PyG Data object containing the graph
        model: Trained SentimentGraphSAGE model
        sample_size: Number of nodes to visualize (for large graphs)
    """
    # Get model predictions
    with torch.no_grad():
        # Get probabilities and predictions
        out = model(graph_data.x.to(device), graph_data.edge_index.to(device))
        preds = out.argmax(dim=1).cpu().numpy()
        probs = F.softmax(out, dim=1).cpu().numpy()

    # Sample nodes if graph is too large
    if len(graph_data.x) > sample_size:
        print(f"Sampling {sample_size} nodes for visualization...")
        indices = torch.randperm(len(graph_data.x))[:sample_size]

        # Create subgraph
        subgraph = graph_data.subgraph(indices)
        preds = preds[indices]
        probs = probs[indices]
    else:
        subgraph = graph_data

    # Convert to NetworkX graph
    G = to_networkx(subgraph, to_undirected=True)

    # Create color map for 3 sentiment classes
    # Define colors: Red = Negative, Gray = Neutral, Green = Positive
    color_map = {
        0: 'red',      # Negative
        1: 'gray',     # Neutral
        2: 'green'     # Positive
    }
    node_colors = [color_map[pred] for pred in preds]

    # Compute layout (force-directed or spectral)
    print("Computing graph layout...")
    try:
        pos = nx.spring_layout(G, k=0.3, iterations=50)  # Force-directed layout
    except:
        pos = nx.spectral_layout(G)  # Fallback for disconnected graphs

    # Draw the graph
    plt.figure(figsize=(16, 12))

    # Draw nodes
    nx.draw_networkx_nodes(
        G, pos,
        node_color=node_colors,
        node_size=50,
        alpha=0.8,
        cmap='viridis'
    )

    # Draw edges
    nx.draw_networkx_edges(
        G, pos,
        edge_color='gray',
        width=0.3,
        alpha=0.2
    )

    # Create legend for sentiment classes
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='green', label='Positive', alpha=0.8),
        Patch(facecolor='gray', label='Neutral', alpha=0.8),
        Patch(facecolor='red', label='Negative', alpha=0.8)
    ]

    plt.legend(handles=legend_elements, loc='upper right', fontsize=10)
    plt.title("Sentiment Analysis Graph\n(Green = Positive, Gray = Neutral, Red = Negative)", fontsize=14)
    plt.axis('off')

    # Add statistics
    sentiment_counts = {}
    for i, class_name in enumerate(graph_data.class_names):
        count = (preds == i).sum()
        sentiment_counts[class_name] = count

    stats_text = f"Total nodes visualized: {len(preds)}\n\nSentiment Distribution:\n"
    for class_name, count in sentiment_counts.items():
        percentage = (count / len(preds)) * 100
        stats_text += f"  {class_name}: {count} ({percentage:.1f}%)\n"

    # Add confidence statistics
    avg_confidences = {}
    for i in range(len(graph_data.class_names)):
        mask = preds == i
        if mask.sum() > 0:
            avg_conf = probs[mask, i].mean()
            avg_confidences[graph_data.class_names[i]] = avg_conf

    stats_text += f"\nAverage Confidence:\n"
    for class_name, conf in avg_confidences.items():
        stats_text += f"  {class_name}: {conf:.2%}\n"

    plt.text(0.02, 0.02, stats_text, transform=plt.gca().transAxes,
            bbox=dict(facecolor='white', alpha=0.9, edgecolor='black'),
            fontsize=9, verticalalignment='bottom')

    plt.tight_layout()
    plt.show()

    # Print numeric summary
    print("\n" + "="*60)
    print("SENTIMENT ANALYSIS VISUALIZATION SUMMARY")
    print("="*60)
    print(f"Total nodes in sample: {len(preds)}")
    print(f"\nPrediction Distribution (Numeric Format):")
    for i, class_name in enumerate(graph_data.class_names):
        count = (preds == i).sum()
        percentage = (count / len(preds)) * 100
        print(f"  Class {i} ({class_name}): {count} nodes ({percentage:.1f}%)")

    return preds, probs

def visualize_sentiment_graph_with_confidence(graph_data, model, sample_size=500):
    """
    Enhanced visualization showing confidence levels with node sizes
    """
    # Get model predictions with confidence
    with torch.no_grad():
        out = model(graph_data.x.to(device), graph_data.edge_index.to(device))
        preds = out.argmax(dim=1).cpu().numpy()
        probs = F.softmax(out, dim=1).cpu().numpy()
        confidences = probs[np.arange(len(preds)), preds]

    # Sample nodes if graph is too large
    if len(graph_data.x) > sample_size:
        print(f"Sampling {sample_size} nodes for visualization...")
        indices = torch.randperm(len(graph_data.x))[:sample_size]
        preds = preds[indices]
        confidences = confidences[indices]
    else:
        indices = np.arange(len(graph_data.x))

    # Create subgraph
    subgraph = graph_data.subgraph(torch.tensor(indices))
    G = to_networkx(subgraph, to_undirected=True)

    # Color map
    color_map = {
        0: 'red',      # Negative
        1: 'gray',     # Neutral
        2: 'green'     # Positive
    }
    node_colors = [color_map[pred] for pred in preds]

    # Node sizes based on confidence (higher confidence = larger node)
    node_sizes = 20 + (confidences * 80)  # Range from 20 to 100

    # Compute layout
    print("Computing graph layout...")
    try:
        pos = nx.spring_layout(G, k=0.3, iterations=50)
    except:
        pos = nx.spectral_layout(G)

    # Draw
    plt.figure(figsize=(16, 12))

    # Draw nodes with varying sizes
    for pred_class in [0, 1, 2]:
        mask = preds == pred_class
        if mask.sum() > 0:
            nx.draw_networkx_nodes(
                G, pos,
                nodelist=list(np.array(list(G.nodes()))[mask]),
                node_color=color_map[pred_class],
                node_size=node_sizes[mask],
                alpha=0.7,
                label=graph_data.class_names[pred_class]
            )

    # Draw edges
    nx.draw_networkx_edges(
        G, pos,
        edge_color='gray',
        width=0.3,
        alpha=0.2
    )

    plt.legend(scatterpoints=1, fontsize=10)
    plt.title("Sentiment Graph Visualization\n(Node size = Prediction Confidence)", fontsize=14)
    plt.axis('off')

    # Add statistics
    stats_text = f"Total nodes: {len(preds)}\n\nAverage Confidence by Class:\n"
    for i, class_name in enumerate(graph_data.class_names):
        mask = preds == i
        if mask.sum() > 0:
            avg_conf = confidences[mask].mean()
            stats_text += f"  {class_name}: {avg_conf:.2%}\n"

    plt.text(0.02, 0.02, stats_text, transform=plt.gca().transAxes,
            bbox=dict(facecolor='white', alpha=0.9, edgecolor='black'),
            fontsize=10, verticalalignment='bottom')

    plt.tight_layout()
    plt.show()

    # Print numeric summary
    print("\n" + "="*60)
    print("CONFIDENCE-BASED VISUALIZATION SUMMARY")
    print("="*60)
    print("Prediction Distribution with Confidence Scores:")
    for i, class_name in enumerate(graph_data.class_names):
        mask = preds == i
        if mask.sum() > 0:
            count = mask.sum()
            percentage = (count / len(preds)) * 100
            avg_conf = confidences[mask].mean()
            print(f"\nClass {i} - {class_name}:")
            print(f"  Count: {count} ({percentage:.1f}%)")
            print(f"  Average Confidence: {avg_conf:.2%}")
            print(f"  Min Confidence: {confidences[mask].min():.2%}")
            print(f"  Max Confidence: {confidences[mask].max():.2%}")

    return preds, confidences

# Example usage
if __name__ == "__main__":
    # Paths
    model_path = "/content/drive/MyDrive/sentiment_gnn_model.pth"
    graph_data_path = "/content/drive/MyDrive/sentiment_graph_data.pkl"

    # Load graph data
    print("Loading graph data...")
    with open(graph_data_path, "rb") as f:
        graph_data = pickle.load(f)

    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Load model
    model = SentimentGraphSAGE(
        in_channels=graph_data.num_features,
        hidden_channels=256,
        num_classes=graph_data.num_classes
    ).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    print("Model loaded successfully!")

    # Basic visualization
    print("\n" + "="*60)
    print("BASIC SENTIMENT GRAPH VISUALIZATION")
    print("="*60)
    visualize_sentiment_graph(graph_data, model, sample_size=1000)

    # Advanced visualization with confidence
    print("\n" + "="*60)
    print("ADVANCED VISUALIZATION WITH CONFIDENCE")
    print("="*60)
    visualize_sentiment_graph_with_confidence(graph_data, model, sample_size=800)

Inference Code

In [ ]:
import pickle
import torch
import numpy as np
import networkx as nx
from sentence_transformers import SentenceTransformer
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
import warnings
warnings.filterwarnings('ignore')

# Define the model class for Sentiment Analysis
class SentimentGraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels=256, num_classes=3):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.bn1 = torch.nn.BatchNorm1d(hidden_channels)

        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.bn2 = torch.nn.BatchNorm1d(hidden_channels)

        self.conv3 = SAGEConv(hidden_channels, hidden_channels // 2)
        self.bn3 = torch.nn.BatchNorm1d(hidden_channels // 2)

        self.fc = torch.nn.Linear(hidden_channels // 2, num_classes)
        self.dropout = torch.nn.Dropout(0.3)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv3(x, edge_index)
        x = self.bn3(x)
        x = F.relu(x)

        x = self.fc(x)
        return x

# Paths
model_path = "/content/drive/MyDrive/sentiment_gnn_model.pth"
graph_data_path = "/content/drive/MyDrive/sentiment_graph_data.pkl"

# Load the stored graph data
print("Loading graph data...")
with open(graph_data_path, "rb") as f:
    graph_data = pickle.load(f)
print("Graph loaded successfully!")
print(f"- Number of nodes: {graph_data.num_nodes}")
print(f"- Number of classes: {graph_data.num_classes}")
print(f"- Class names: {graph_data.class_names}")

# Load the trained model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

model = SentimentGraphSAGE(
    in_channels=graph_data.num_features,
    hidden_channels=256,
    num_classes=graph_data.num_classes
).to(device)

# Load model weights
try:
    model.load_state_dict(torch.load(model_path, map_location=device))
    print("Model loaded successfully!")
except FileNotFoundError:
    print(f"Model file not found at {model_path}")
    print("Please check the path and try again.")
    exit(1)

model.eval()

# Load the Sentence Transformer model
embed_model = SentenceTransformer("all-mpnet-base-v2", device=device)

# Function to predict sentiment for a given sentence
def predict_sentiment(sentence):
    """
    Predict sentiment for a single sentence
    Returns: predicted sentiment label (Positive/Negative/Neutral)
    """
    # Convert the sentence into an embedding
    sentence_embedding = embed_model.encode(sentence, convert_to_tensor=True).cpu().numpy()

    # Find the most similar node in the graph using cosine similarity
    node_embeddings = graph_data.x.cpu().numpy()

    # Calculate cosine similarities
    norms = np.linalg.norm(node_embeddings, axis=1)
    sentence_norm = np.linalg.norm(sentence_embedding)
    similarities = np.dot(node_embeddings, sentence_embedding) / (norms * sentence_norm + 1e-8)

    # Get the most similar node
    closest_node_idx = np.argmax(similarities)

    # Perform model prediction
    with torch.no_grad():
        # Get predictions for all nodes
        out = model(graph_data.x.to(device), graph_data.edge_index.to(device))

        # Get prediction for the most similar node
        node_logits = out[closest_node_idx]
        node_probs = F.softmax(node_logits, dim=0).cpu().numpy()

    # Get prediction
    final_pred_class = np.argmax(node_probs)
    final_probs = node_probs

    # Map to sentiment labels
    sentiment_labels = graph_data.class_names
    predicted_sentiment = sentiment_labels[final_pred_class]
    confidence = final_probs[final_pred_class] * 100

    # Get all class probabilities
    class_probabilities = {sentiment_labels[i]: final_probs[i]*100 for i in range(len(sentiment_labels))}

    print("\n" + "="*60)
    print("SENTIMENT ANALYSIS PREDICTION RESULT")
    print("="*60)
    print(f"Input Sentence: {sentence}")
    print(f"\nPredicted Sentiment: {predicted_sentiment.upper()}")
    print(f"Confidence: {confidence:.2f}%")
    print(f"\nClass Probabilities:")

    # Sort by probability
    sorted_probs = sorted(class_probabilities.items(), key=lambda x: x[1], reverse=True)
    for sentiment, prob in sorted_probs:
        bar_length = int(prob / 5)
        bar = "█" * bar_length + "░" * (20 - bar_length)
        print(f"  {sentiment:8}: {prob:6.2f}% {bar}")

    print("="*60)

    return predicted_sentiment, final_probs, confidence

# Test sentences
if __name__ == "__main__":
    print("\n" + "="*60)
    print("SENTIMENT ANALYSIS INFERENCE")
    print("="*60)

    # Test with negative sentence
    test_sentence = "I ordered a kurta from this seller three weeks ago, still hasn't arrived and nobody is responding to my emails"
    predict_sentiment(test_sentence)


In [ ]:
import pickle
import torch
import numpy as np
import networkx as nx
from sentence_transformers import SentenceTransformer
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
import warnings
warnings.filterwarnings('ignore')

# Define the model class for Sentiment Analysis
class SentimentGraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels=256, num_classes=3):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.bn1 = torch.nn.BatchNorm1d(hidden_channels)

        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.bn2 = torch.nn.BatchNorm1d(hidden_channels)

        self.conv3 = SAGEConv(hidden_channels, hidden_channels // 2)
        self.bn3 = torch.nn.BatchNorm1d(hidden_channels // 2)

        self.fc = torch.nn.Linear(hidden_channels // 2, num_classes)
        self.dropout = torch.nn.Dropout(0.3)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv3(x, edge_index)
        x = self.bn3(x)
        x = F.relu(x)

        x = self.fc(x)
        return x

# Paths
model_path = "/content/drive/MyDrive/sentiment_gnn_model.pth"
graph_data_path = "/content/drive/MyDrive/sentiment_graph_data.pkl"

# Load the stored graph data
print("Loading graph data...")
with open(graph_data_path, "rb") as f:
    graph_data = pickle.load(f)
print("Graph loaded successfully!")
print(f"- Number of nodes: {graph_data.num_nodes}")
print(f"- Number of classes: {graph_data.num_classes}")
print(f"- Class names: {graph_data.class_names}")

# Load the trained model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

model = SentimentGraphSAGE(
    in_channels=graph_data.num_features,
    hidden_channels=256,
    num_classes=graph_data.num_classes
).to(device)

# Load model weights
try:
    model.load_state_dict(torch.load(model_path, map_location=device))
    print("Model loaded successfully!")
except FileNotFoundError:
    print(f"Model file not found at {model_path}")
    print("Please check the path and try again.")
    exit(1)

model.eval()

# Load the Sentence Transformer model
embed_model = SentenceTransformer("all-mpnet-base-v2", device=device)

# Function to predict sentiment for a given sentence
def predict_sentiment(sentence):
    """
    Predict sentiment for a single sentence
    Returns: predicted sentiment label (Positive/Negative/Neutral)
    """
    # Convert the sentence into an embedding
    sentence_embedding = embed_model.encode(sentence, convert_to_tensor=True).cpu().numpy()

    # Find the most similar node in the graph using cosine similarity
    node_embeddings = graph_data.x.cpu().numpy()

    # Calculate cosine similarities
    norms = np.linalg.norm(node_embeddings, axis=1)
    sentence_norm = np.linalg.norm(sentence_embedding)
    similarities = np.dot(node_embeddings, sentence_embedding) / (norms * sentence_norm + 1e-8)

    # Get the most similar node
    closest_node_idx = np.argmax(similarities)

    # Perform model prediction
    with torch.no_grad():
        # Get predictions for all nodes
        out = model(graph_data.x.to(device), graph_data.edge_index.to(device))

        # Get prediction for the most similar node
        node_logits = out[closest_node_idx]
        node_probs = F.softmax(node_logits, dim=0).cpu().numpy()

    # Get prediction
    final_pred_class = np.argmax(node_probs)
    final_probs = node_probs

    # Map to sentiment labels
    sentiment_labels = graph_data.class_names
    predicted_sentiment = sentiment_labels[final_pred_class]
    confidence = final_probs[final_pred_class] * 100

    # Get all class probabilities
    class_probabilities = {sentiment_labels[i]: final_probs[i]*100 for i in range(len(sentiment_labels))}

    print("\n" + "="*60)
    print("SENTIMENT ANALYSIS PREDICTION RESULT")
    print("="*60)
    print(f"Input Sentence: {sentence}")
    print(f"\nPredicted Sentiment: {predicted_sentiment.upper()}")
    print(f"Confidence: {confidence:.2f}%")
    print(f"\nClass Probabilities:")

    # Sort by probability
    sorted_probs = sorted(class_probabilities.items(), key=lambda x: x[1], reverse=True)
    for sentiment, prob in sorted_probs:
        bar_length = int(prob / 5)
        bar = "█" * bar_length + "░" * (20 - bar_length)
        print(f"  {sentiment:8}: {prob:6.2f}% {bar}")

    print("="*60)

    return predicted_sentiment, final_probs, confidence

# Test sentences
if __name__ == "__main__":
    print("\n" + "="*60)
    print("SENTIMENT ANALYSIS INFERENCE")
    print("="*60)

    # Test with negative sentence
    test_sentence = "I would like to share my positive experience, as the yoga instructor bridgenton taught breathing techniques that significantly helped in reducing my anxiety and improved my overall well-being."
    predict_sentiment(test_sentence)
